# Data audit and historical split reconstruction

Scans/hashes the real images, screens duplicates, derives audit proxy bins and reconstructs the released seven-fold split. No training.

Run the single code cell below with the **Python (skin)** kernel.

In [ ]:
from pathlib import Path
import sys

PACKAGE_DIR_INPUT = Path(r"D:\AIUB\Neural\A_Comprehensive_Framework_for_Auditing_Demographic_Bias_in_Synthetic_Skin_Disease_Datasets\revision_package")
if not PACKAGE_DIR_INPUT.is_dir():
    raise FileNotFoundError(f"revision_package not found: {PACKAGE_DIR_INPUT}")
if str(PACKAGE_DIR_INPUT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR_INPUT))

import json
import pandas as pd

from notebook_config import DATA_ROOT, PHASH_DISTANCE, RUN_ROOT
from notebook_support import (
    run_once,
    select_metadata,
    validate_hierarchical_image_mapping,
)

metadata_path = select_metadata(require_known_hash=True)
mapping = validate_hierarchical_image_mapping(metadata_path)
print("Validated dataset mapping:", mapping)

audit_dir = run_once(
    "02_audit_data.py",
    [
        "--data-root", str(DATA_ROOT),
        "--metadata", str(metadata_path),
        "--reconstruct-legacy-splits",
        "--confirm-original-row-order",
        "--phash-distance", str(PHASH_DISTANCE),
    ],
    RUN_ROOT / "data_audit",
    [
        "audit_metadata.csv",
        "image_inventory.csv",
        "duplicate_candidates_for_review.csv",
        "reconstructed_fold_manifest.csv",
        "duplicate_candidates_crossing_roles.csv",
        "dataset_accounting.csv",
        "audit_notes.json",
    ],
)

audited = pd.read_csv(audit_dir / "audit_metadata.csv")
pairs = pd.read_csv(audit_dir / "duplicate_candidates_for_review.csv")
crossing = pd.read_csv(audit_dir / "duplicate_candidates_crossing_roles.csv")
notes = json.loads((audit_dir / "audit_notes.json").read_text(encoding="utf-8"))

if len(pairs):
    status = pairs["verified_related"].fillna("").astype(str).str.strip().str.lower()
    exact_pixels = pairs["pixel_identical"].astype(str).str.lower().eq("true")
    metadata_ids = set(audited["sample_id"].astype(str))
    in_cohort = (
        pairs["sample_id_a"].astype(str).isin(metadata_ids)
        & pairs["sample_id_b"].astype(str).isin(metadata_ids)
    )
    cohort_pairs = pairs.loc[in_cohort].copy()
    cohort_status = cohort_pairs["verified_related"].fillna("").astype(str).str.strip().str.lower()
    cross_role_unique_pairs = int(
        crossing[["sample_id_a", "sample_id_b"]].drop_duplicates().shape[0]
    )
else:
    exact_pixels = pd.Series(dtype=bool)
    cohort_pairs = pairs.copy()
    cohort_status = pd.Series(dtype=str)
    cross_role_unique_pairs = 0

summary = {
    "metadata_rows": int(len(audited)),
    "age_proxy_counts": audited["age_proxy"].value_counts(dropna=False).to_dict(),
    "gender_proxy_counts": audited["gender_proxy"].value_counts(dropna=False).to_dict(),
    "appearance_proxy_counts": audited["appearance_proxy"].value_counts(dropna=False).to_dict(),
    "archive_wide_candidate_pairs": int(len(pairs)),
    "archive_wide_exact_pixel_pairs": int(exact_pixels.sum()),
    "cohort_candidate_pairs": int(len(cohort_pairs)),
    "cohort_exact_pixel_pairs": int(
        cohort_pairs["pixel_identical"].astype(str).str.lower().eq("true").sum()
    ) if len(cohort_pairs) else 0,
    "cohort_unresolved_candidate_pairs": int(
        (~cohort_status.isin(["yes", "no"])).sum()
    ),
    "cross_role_candidate_rows": int(len(crossing)),
    "cross_role_unique_candidate_pairs": cross_role_unique_pairs,
    "missing_identity_fields": notes.get("missing_id_fields", []),
    "output_directory": str(audit_dir),
}
print(json.dumps(summary, indent=2, default=str))
print("\nNo neural model was trained. Archive-wide and cohort candidates are reported separately.")
print("Run notebook 04 to inspect only pairs for which BOTH images are in the 565-image cohort.")
